# Vessel Segmentation — Interactive Overlay Viewer

Visualize raw fluorescence image superimposed with binary vessel mask.

- **Dropdown** — select matched image by ID
- **Slider** — control mask overlay transparency

In [ ]:
import glob, os
import numpy as np
import matplotlib.pyplot as plt
from tifffile import imread
from ipywidgets import interact, FloatSlider

# ── EDIT THESE ────────────────────────────────────────
INPUT_DIR  = '../Input/D7'
OUTPUT_DIR = '../Output/D7'
# ──────────────────────────────────────────────────────

def load_by_id(folder):
    files = glob.glob(os.path.join(folder, '*.tif'))
    return {int(os.path.basename(f).split('_')[0]): f for f in files}

inputs  = load_by_id(INPUT_DIR)
outputs = load_by_id(OUTPUT_DIR)
matched = sorted(set(inputs) & set(outputs))
print(f'Matched IDs: {matched}')

In [ ]:
def normalize(img):
    img = img.astype(float)
    return (img - img.min()) / (img.max() - img.min() + 1e-8)

def show(image_id, alpha):
    img  = imread(inputs[image_id])                   # (H, W)  single channel uint16
    mask = imread(outputs[image_id]) > 0              # (H, W)  bool · 255=lumen→True

    # Info
    h, w       = img.shape[:2]
    dtype      = img.dtype
    vmin, vmax = img.min(), img.max()
    n_vessel   = mask.sum()
    pct_vessel = 100 * n_vessel / mask.size
    fname_in   = os.path.basename(inputs[image_id])
    fname_out  = os.path.basename(outputs[image_id])

    fig, ax = plt.subplots(figsize=(14, 14))

    # Layer 1: raw grayscale image — single channel, no RGB conversion
    ax.imshow(normalize(img), cmap='gray')

    # Layer 2: mask as RGBA — alpha controls transparency directly
    mask_rgba          = np.zeros((*mask.shape, 4))   # (H, W, 4)  all transparent
    mask_rgba[mask]    = [1, 0, 0, alpha]             # vessel pixels → red
    ax.imshow(mask_rgba)

    ax.set_title(
        f'Input:  {fname_in}\n'
        f'Output: {fname_out}\n'
        f'Resolution: {w} x {h} px  |  dtype: {dtype}  |  '
        f'Intensity range: [{vmin}, {vmax}]  |  '
        f'Vessel pixels: {n_vessel:,} ({pct_vessel:.1f}%)  |  '
        f'Mask alpha: {alpha:.2f}',
        fontsize=11, loc='left'
    )
    ax.axis('off')
    plt.tight_layout()
    plt.show()

interact(
    show,
    image_id = matched,
    alpha    = FloatSlider(value=0.4, min=0.0, max=1.0, step=0.05,
                           description='Mask alpha', continuous_update=False)
);